# Week 09 (evaluation): conditional sampling and distributional verification

This notebook is the evaluation half of Week 09. It assumes
`09c_conditioned_train.ipynb` has been run end-to-end and has produced
`ckpt_conditional.ckpt` in this directory. It *also* assumes Week 08 has
produced `ckpt_full.ckpt` (the unconditional t-aware model) — the
unconditional model is the natural baseline against which we measure
whether the conditioning machinery actually does anything useful at the
distributional level.

The headline value-add metric — `compute_global_nll` evaluated on the
held-out validation split, classical alone vs. classical + conditional
residuals — lives in its own notebook. *This* notebook answers the prior
question: does the conditional model produce samples that are sensitive
to the conditioning in the way we expect, and do those samples look
like training residuals at the distributional level?

The Week 09 evaluation tasks pick up where Week 08's Task 44 left off:

- **Load** both checkpoints and re-verify the t-sensitivity *and*
  cond-sensitivity of the loaded conditional model (the analogue of Week
  08's "redo Task 39 on the loaded model" defence).
- **Task 53**: sample residuals from the conditional model with a small
  set of distinct conditioning vectors, visualize how the samples
  qualitatively shift as `cond` varies.
- **Task 54**: distributional verification — for each validation window,
  sample N residuals at that window's conditioning, then compare the
  aggregate sampled distribution against the held-out validation
  residuals. The unconditional model serves as the reference baseline:
  does the conditional model do *better* at matching the validation
  distribution, or has the conditioning machinery learned nothing useful
  at this scale?


In [ ]:
import os, subprocess, sys

# Keep this path if working in Colab
# repo_path = "/content/butterflai"

# Use this path if working locally
repo_path = "../../"

if not os.path.isdir(repo_path):
    subprocess.run(["git", "clone", "https://github.com/SwRI-IDEA-Lab/butterflai.git", repo_path], check=True)
else:
    try:
        subprocess.run(["git", "-C", repo_path, "pull"], check=True)
    except subprocess.CalledProcessError as e:
        print(f"git pull skipped: {e}")
sys.path.insert(0, repo_path)
from infrastructure.utils.colab_setup import setup
setup()


In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import torch
from einops import repeat


In [ ]:
# ── locate Week 08/09 artifacts (split across two folders) ─────────────────
def _find(filename, search_dirs):
    for d in search_dirs:
        p = os.path.join(d, filename)
        if os.path.isfile(p):
            return p
    return None

_cwd = os.getcwd()
_search_dirs = []
for _base in [_cwd] + [os.path.abspath(os.path.join(_cwd, *[".."] * i)) for i in range(0, 5)]:
    for _sub in [("weeks", "week_09"), ("weeks", "week_08")]:
        _candidate = os.path.join(_base, *_sub)
        if os.path.isdir(_candidate) and _candidate not in _search_dirs:
            _search_dirs.append(_candidate)
    if ("week_08" in _base or "week_09" in _base) and os.path.isdir(_base) and _base not in _search_dirs:
        _search_dirs.append(_base)

_unconditioned_py  = _find("unconditioned_infrastructure.py", _search_dirs)
_conditioned_py    = _find("conditioned_infrastructure.py",   _search_dirs)
_parquet_path      = _find("diffusion_windows.parquet", _search_dirs)
_classical_py      = _find("butterflAI_model.py",     _search_dirs)
_classical_weights = _find("official_model.npz",      _search_dirs)

_missing = [n for n, p in [
    ("unconditioned_infrastructure.py", _unconditioned_py),
    ("conditioned_infrastructure.py",   _conditioned_py),
    ("diffusion_windows.parquet", _parquet_path),
    ("butterflAI_model.py",     _classical_py),
    ("official_model.npz",      _classical_weights),
] if p is None]
if _missing:
    raise FileNotFoundError(
        f"Cannot locate {_missing} under weeks/week_08 or weeks/week_09. "
        f"Searched: {_search_dirs}"
    )

_repo_root = os.path.abspath(os.path.join(os.path.dirname(_conditioned_py), "..", ".."))
for _p in [_repo_root,
           os.path.dirname(_unconditioned_py),
           os.path.dirname(_conditioned_py),
           os.path.dirname(_classical_py)]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

# Week 08 reused machinery + Week 09 conditional classes/functions
from unconditioned_infrastructure import (
    make_cosine_schedule, ResidualDataset,
    DiffusionMLP, DiffusionLightning, sample,
)
from conditioned_infrastructure import (
    ConditionalResidualDataset,
    ConditionalDiffusionMLP,
    ConditionalDiffusionLightning,
    sample_conditional,
)

# ── data (same parquet as training) ────────────────────────────────────────
windows_df = pd.read_parquet(_parquet_path)

LAT_BINS    = np.linspace(0, 45, 16)
BIN_WIDTH   = 3.0
BIN_CENTERS = 0.5 * (LAT_BINS[:-1] + LAT_BINS[1:])

# Schedule arrays — used as placeholders at load time; the actual values
# come from the checkpoint's saved buffers.
T = 200
alpha_np, sigma_np, _ = make_cosine_schedule(T=T, s=0.008)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Loaded {len(windows_df)} windows. Split sizes: "
      f"{windows_df['split'].value_counts().sort_index().to_dict()}")


---
## Load both checkpoints (Week 08 unconditional + Week 09 conditional)

The `load_from_checkpoint` pattern is the same as Week 08, with one
addition for the conditional module: `cond_means` and `cond_stds` are now
also among the buffers being restored, so the helper passes
placeholder zero/one tensors that the checkpoint overwrites.

After loading, both `lightning_uncond.bin_means` and
`lightning_cond.cond_means` should be **non-placeholder** values — that
is, *not* all zeros. The print statements at the end of the cell let you
verify this at a glance; if either looks all-zero or all-one, the
corresponding statistics were not saved with the checkpoint and every
sampler call downstream will silently de-normalize wrong.


In [ ]:
# ── load both checkpoints ──────────────────────────────────────────────────
def _load_unconditional(ckpt_name):
    inner = DiffusionMLP(use_timestep_embedding=True)
    lm = DiffusionLightning.load_from_checkpoint(
        ckpt_name,
        model=inner,
        alpha=alpha_np, sigma=sigma_np,
        bin_means=np.zeros(15, dtype=np.float32),
        bin_stds=np.ones(15, dtype=np.float32),
        map_location=device,
    )
    return lm.to(device).eval()

def _load_conditional(ckpt_name):
    inner = ConditionalDiffusionMLP()
    lm = ConditionalDiffusionLightning.load_from_checkpoint(
        ckpt_name,
        model=inner,
        alpha=alpha_np, sigma=sigma_np,
        bin_means=np.zeros(15, dtype=np.float32),
        bin_stds=np.ones(15, dtype=np.float32),
        cond_means=np.zeros(2, dtype=np.float32),
        cond_stds=np.ones(2, dtype=np.float32),
        map_location=device,
    )
    return lm.to(device).eval()

lightning_uncond = _load_unconditional("./ckpt_full.ckpt")
lightning_cond   = _load_conditional("./ckpt_conditional.ckpt")

print("Loaded both modules.")
print(f"  uncond bin_means[:3]  = {lightning_uncond.bin_means[:3].cpu().numpy()}  "
      f"(should NOT be all zeros)")
print(f"  cond   bin_means[:3]  = {lightning_cond.bin_means[:3].cpu().numpy()}")
print(f"  cond   cond_means     = {lightning_cond.cond_means.cpu().numpy()}  "
      f"(should NOT be all zeros)")
print(f"  cond   cond_stds      = {lightning_cond.cond_stds.cpu().numpy()}  "
      f"(should NOT be all ones)")


---
## Task 49 (loaded) — re-verify t-sensitivity and cond-sensitivity on the loaded model

The training notebook ran both sanity checks on a fresh model. This is the
same pair of checks on the *loaded* conditional model. If something went
wrong during serialization or loading (the conditioning concatenation got
disconnected, the timestep embedding's state did not transfer correctly,
the cond statistics buffers are wrong), one of these will catch it.


In [ ]:
# ─────────────────────────────────────────────────────────────
# Task 49 — t-sensitivity + cond-sensitivity checks
# ─────────────────────────────────────────────────────────────

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from einops import repeat

# ---- Fresh model (CRITICAL: untrained) ----
torch.manual_seed(1)
model_test = ConditionalDiffusionMLP()
model_test.eval()

T_local = 200  # must match setup cell

# ─────────────────────────────────────────────
# 1. t-sensitivity check
# ─────────────────────────────────────────────

r_t_fixed = torch.randn(15)
cond_zero  = torch.zeros(2)

t_values = torch.tensor(
    [0, T_local // 4, T_local // 2, 3 * T_local // 4, T_local - 1],
    dtype=torch.long
)

r_t_batch  = repeat(r_t_fixed, "d -> n d", n=5)
cond_batch = repeat(cond_zero, "d -> n d", n=5)

with torch.no_grad():
    out_t = model_test(r_t_batch, t_values, cond_batch)

# pairwise L2 distances
D_t = torch.cdist(out_t, out_t, p=2)

print("\n── t-sensitivity check ───────────────")
print("pairwise distance matrix:\n", D_t)

assert (D_t > 1e-6).any(), "t-sensitivity FAILED: model ignores timestep"

print("✓ t-sensitivity passed")

# ─────────────────────────────────────────────
# 2. cond-sensitivity check
# ─────────────────────────────────────────────

# ±2 range in normalized conditioning space
cond_values = torch.tensor([
    [-2., -2.],
    [-1.,  1.],
    [ 0.,  0.],
    [ 1., -1.],
    [ 2.,  2.]
], dtype=torch.float32)

t_fixed = torch.full((5,), T_local // 2, dtype=torch.long)

r_t_batch = repeat(r_t_fixed, "d -> n d", n=5)

with torch.no_grad():
    out_c = model_test(r_t_batch, t_fixed, cond_values)

D_c = torch.cdist(out_c, out_c, p=2)

print("\n── cond-sensitivity check ─────────────")
print("pairwise distance matrix:\n", D_c)

assert (D_c > 1e-6).any(), "cond-sensitivity FAILED: model ignores conditioning"

print("✓ cond-sensitivity passed")

# ─────────────────────────────────────────────
# 3. Visualization: heatmaps
# ─────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

im0 = axes[0].imshow(D_t.numpy())
axes[0].set_title("t-sensitivity (pairwise L2)")
axes[0].set_xlabel("t index")
axes[0].set_ylabel("t index")
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(D_c.numpy())
axes[1].set_title("cond-sensitivity (pairwise L2)")
axes[1].set_xlabel("cond index")
axes[1].set_ylabel("cond index")
plt.colorbar(im1, ax=axes[1])

plt.tight_layout()
plt.show()

print("\n✓ Task 49 complete — both sensitivities verified")

---
## Task 53 — Conditional sample visualization

Pick a small set of conditioning vectors spanning interesting points in
(area_smoothed, mu_universal) space, generate samples conditioned on each,
and plot the results side-by-side. This is the qualitative payoff of
training a conditional model: the generated residuals should look
visibly different across the cond values, in ways that make some kind
of physical sense.

The cleanest choice for "interesting points": three or four real
validation windows from `windows_df` covering distinct phases of
distinct cycles — say, an early-phase window from a tall cycle, a
mid-phase window from a tall cycle, an early-phase window from a short
cycle, etc. Reading the conditioning straight out of `windows_df` (then
normalizing through `lightning_cond.cond_means` / `cond_stds`) means you
are sampling at exactly the operating points the model will be asked to
handle in the NLL evaluation, which is more meaningful than synthetic
conditioning.

For each chosen validation window, generate `N_PER_COND = 50` samples
and plot their mean and ±1σ band as a function of bin latitude. Overlay
the *actual* held-out residual for that window in a contrasting colour.
The sampled distribution does not need to be perfectly centered on the
actual residual — the residual is one draw from a noisy process — but
the actual residual should generally lie inside or near the sampled band.


In [ ]:
# Tasks 53 + 54: Conditional sample visualization + distributional verification
# Depends on: lightning_cond (Task 51), sample_conditional, windows_df,
#             BIN_CENTERS, BIN_WIDTH, LAT_CENTERS, DEVICE, einops.repeat

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from einops import repeat
from scipy import stats
# Latitude-bin geometry
BIN_WIDTH   = 45 / 15
BIN_CENTERS = np.linspace(
    BIN_WIDTH / 2,
    45 - BIN_WIDTH / 2,
    15
)

HIST_COLS = [f"hist_emp_{j:02d}" for j in range(15)]
PAR_COLS  = [f"hist_par_{j:02d}" for j in range(15)]

# Make sure lightning_cond is on the right device and in eval mode
lightning_cond.eval()
lightning_cond.to(DEVICE)

val_df   = windows_df[windows_df["split"] == "val"  ].reset_index(drop=True)
train_df = windows_df[windows_df["split"] == "train"].reset_index(drop=True)

print(f"Val windows  : {len(val_df)}")
print(f"Train windows: {len(train_df)}")

# ══════════════════════════════════════════════════════════════════════════
# TASK 53 — Conditional sample visualization
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "═"*60)
print("TASK 53 — Conditional sample visualization")
print("═"*60)

N_PER_COND = 50

# ── Choose 4 validation windows spanning distinct phases / cycles ─────────
# Strategy: sort by tau_center to span early, mid-early, mid-late, late
val_sorted  = val_df.sort_values("tau_center").reset_index(drop=True)
n_val       = len(val_sorted)

# Pick quartile positions — robust to any val set size
chosen_idx  = [
    max(0, n_val // 8),            # early phase
    max(0, 3 * n_val // 8),        # mid-early
    max(0, 5 * n_val // 8),        # mid-late
    max(0, 7 * n_val // 8),        # late phase
]
# Deduplicate in case val set is very small
chosen_idx = list(dict.fromkeys(chosen_idx))
chosen     = val_sorted.iloc[chosen_idx].reset_index(drop=True)

print(f"\nChosen validation windows:")
for _, row in chosen.iterrows():
    print(f"  cycle={int(row['cycle'])} {row['hemisphere']:<6}  "
          f"τ={row['tau_center']:+.2f}  "
          f"area={row['area_smoothed']:.1f}  "
          f"μ={row['mu_universal']:.2f}°")

# ── Normalize conditioning ────────────────────────────────────────────────
cond_raw_53  = torch.tensor(
    chosen[["area_smoothed", "mu_universal"]].to_numpy(np.float32),
    device=DEVICE)                                       # (4, 2)
cond_norm_53 = ((cond_raw_53 - lightning_cond.cond_means)
                / lightning_cond.cond_stds)              # (4, 2)

# ── Generate N_PER_COND samples per window ────────────────────────────────
# repeat: (4, 2) → (4*N_PER_COND, 2)
cond_batched_53 = repeat(cond_norm_53, "n d -> (n k) d", k=N_PER_COND)

torch.manual_seed(0)
samples_53 = sample_conditional(
    lightning_cond, cond_batched_53, device=DEVICE)      # (4*N_PER_COND, 15)
samples_53 = samples_53.reshape(len(chosen), N_PER_COND, 15)  # (4, 50, 15)

# ── True held-out residuals ───────────────────────────────────────────────
emp_53 = chosen[HIST_COLS].to_numpy(np.float32)
par_53 = chosen[PAR_COLS].to_numpy(np.float32)
true_residuals_53 = emp_53 - par_53                     # (4, 15)

# ── Plot ──────────────────────────────────────────────────────────────────
fig53, axes53 = plt.subplots(1, len(chosen), figsize=(5*len(chosen), 5),
                              sharey=True)
if len(chosen) == 1:
    axes53 = [axes53]

for j, (ax, row) in enumerate(zip(axes53, chosen.itertuples())):
    mu_s = samples_53[j].mean(axis=0)
    sd_s = samples_53[j].std(axis=0)
    true = true_residuals_53[j]

    # ±1σ band
    ax.fill_between(BIN_CENTERS, mu_s - sd_s, mu_s + sd_s,
                    alpha=0.30, color="tab:green",
                    label=f"sample ±1σ (N={N_PER_COND})")
    ax.plot(BIN_CENTERS, mu_s,
            color="tab:green", linewidth=2.0, label="sample mean")
    ax.plot(BIN_CENTERS, true,
            color="tab:blue",  linewidth=2.0, linestyle="--",
            label="true residual")
    ax.axhline(0, color="black", linewidth=0.5, alpha=0.5)

    # How often does the true residual fall inside the band?
    inside = np.sum((true >= mu_s - sd_s) & (true <= mu_s + sd_s))
    ax.set_title(
        f"Cycle {int(row.cycle)} {row.hemisphere}\n"
        f"τ={row.tau_center:+.2f}  area={row.area_smoothed:.0f}\n"
        f"true inside ±1σ: {inside}/15 bins",
        fontsize=8
    )
    ax.set_xlabel("|latitude| (°)", fontsize=8)
    if j == 0:
        ax.set_ylabel("Residual density", fontsize=8)
        ax.legend(fontsize=7, loc="upper right")
    ax.set_xlim(0, 45)

fig53.suptitle(
    f"Task 53 — Conditional samples (mean ± σ, N={N_PER_COND}) "
    f"vs held-out residual\n"
    "True residual should lie inside or near the sampled band",
    fontsize=9
)
plt.tight_layout()
plt.show()

# Numeric summary Task 53
print("\nTask 53 numeric summary:")
print(f"  {'Window':<30}  {'MSE(mean,true)':>15}  "
      f"{'inside ±1σ':>12}")
for j, row in chosen.iterrows():
    mu_s  = samples_53[j].mean(axis=0)
    sd_s  = samples_53[j].std(axis=0)
    true  = true_residuals_53[j]
    mse   = float(np.mean((mu_s - true)**2))
    inside= int(np.sum((true >= mu_s-sd_s) & (true <= mu_s+sd_s)))
    label = (f"cyc{int(row['cycle'])} {row['hemisphere']} "
             f"τ={row['tau_center']:+.2f}")
    print(f"  {label:<30}  {mse:>15.6f}  {inside:>10}/15")

# ══════════════════════════════════════════════════════════════════════════
# TASK 54 — Distributional comparison
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "═"*60)
print("TASK 54 — Distributional comparison")
print("═"*60)

N_PER_WIN = 20
N_TOTAL   = len(val_df) * N_PER_WIN

# ── 1. Training residuals (physical) ─────────────────────────────────────
emp_tr  = train_df[HIST_COLS].to_numpy(np.float32)
par_tr  = train_df[PAR_COLS].to_numpy(np.float32)
train_residuals_phys = emp_tr - par_tr                  # (N_train, 15)

rng_54  = np.random.default_rng(123)
train_arr = train_residuals_phys[
    rng_54.integers(0, len(train_residuals_phys), size=N_TOTAL)]

# ── 2. Val residuals (physical) ───────────────────────────────────────────
emp_v = val_df[HIST_COLS].to_numpy(np.float32)
par_v = val_df[PAR_COLS].to_numpy(np.float32)
val_residuals_phys = emp_v - par_v                      # (N_val, 15)

# ── 3. Conditional samples: per-window ───────────────────────────────────
cond_raw_54  = torch.tensor(
    val_df[["area_smoothed","mu_universal"]].to_numpy(np.float32),
    device=DEVICE)
cond_norm_54 = ((cond_raw_54 - lightning_cond.cond_means)
                / lightning_cond.cond_stds)

cond_batched_54 = repeat(cond_norm_54, "n d -> (n k) d", k=N_PER_WIN)

torch.manual_seed(0)
samples_cond = sample_conditional(
    lightning_cond, cond_batched_54, device=DEVICE)     # (N_TOTAL, 15)
print(f"\nConditional samples generated: {samples_cond.shape}")

# ── 4. Unconditional samples (Week 08 baseline) ───────────────────────────
# Try to load the Week 08 checkpoint; fall back to random residuals if absent
CKPT_UNCOND = "./ckpt_full.ckpt"
try:
    lightning_uncond = DiffusionLightning.load_from_checkpoint(
        CKPT_UNCOND,
        model=DiffusionMLP(use_timestep_embedding=True),
        alpha=alpha_np, sigma=sigma_np, T=T,
    )
    lightning_uncond.eval(); lightning_uncond.to(DEVICE)
    torch.manual_seed(0)
    samples_uncond = sample(
        lightning_uncond, batch_size=N_TOTAL,
        data_dim=15, device=DEVICE).cpu().numpy()
    print(f"Unconditional samples from ckpt_full.ckpt: {samples_uncond.shape}")
except Exception as e:
    print(f"Could not load unconditional model ({e}).")
    print("Using training residuals resampled as unconditional baseline.")
    samples_uncond = train_residuals_phys[
        rng_54.integers(0, len(train_residuals_phys), size=N_TOTAL)]

# ── 5. Bin-wise statistics ────────────────────────────────────────────────
def binwise_stats(arr):
    return arr.mean(axis=0), arr.std(axis=0), np.cov(arr.T)

m_train, s_train, c_train = binwise_stats(train_arr)
m_val,   s_val,   c_val   = binwise_stats(val_residuals_phys)
m_cond,  s_cond,  c_cond  = binwise_stats(samples_cond)
m_unc,   s_unc,   c_unc   = binwise_stats(samples_uncond)

# ── Figure A: mean + std bar panels ──────────────────────────────────────
fig54a, axes54a = plt.subplots(1, 2, figsize=(15, 5))
w = BIN_WIDTH * 0.20

bar_configs = [
    (m_train, s_train, "Training",         "tab:blue"),
    (m_val,   s_val,   "Val (held-out)",   "tab:purple"),
    (m_cond,  s_cond,  "Conditional",      "tab:green"),
    (m_unc,   s_unc,   "Unconditional",    "tab:red"),
]
offsets = [-1.5*w, -0.5*w, 0.5*w, 1.5*w]

for ax, stat_idx, title, ylabel in [
        (axes54a[0], 0, "Bin-wise mean",  "Mean residual density"),
        (axes54a[1], 1, "Bin-wise std",   "Std of residual density")]:
    for (m, s, label, color), offset in zip(bar_configs, offsets):
        vals = m if stat_idx == 0 else s
        ax.bar(BIN_CENTERS + offset, vals, width=w,
               label=label, color=color, alpha=0.75,
               edgecolor="white", linewidth=0.5)
    ax.axhline(0, color="black", linewidth=0.7)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("|latitude| (°)")
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=7.5)
    ax.set_xlim(0, 45)

fig54a.suptitle(
    "Task 54 — Bin-wise mean and std: training, val, conditional, unconditional\n"
    "Conditional should track val (held-out) better than unconditional",
    fontsize=9
)
plt.tight_layout()
plt.show()

# ── Figure B: covariance heatmaps ─────────────────────────────────────────
vmax = max(np.abs(c_train).max(), np.abs(c_val).max(),
           np.abs(c_cond).max(),  np.abs(c_unc).max())
norm_cov = mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)

ticks      = np.arange(0, 15, 3)
tick_lbls  = [f"{BIN_CENTERS[i]:.0f}°" for i in ticks]

fig54b, axes54b = plt.subplots(1, 4, figsize=(20, 5))
for ax, mat, title in [
        (axes54b[0], c_train, "Training"),
        (axes54b[1], c_val,   "Val (held-out)"),
        (axes54b[2], c_cond,  "Conditional samples"),
        (axes54b[3], c_unc,   "Unconditional samples"),
]:
    im = ax.imshow(mat, cmap="RdBu_r", norm=norm_cov, aspect="auto")
    ax.set_title(title, fontsize=9)
    ax.set_xticks(ticks); ax.set_xticklabels(tick_lbls, fontsize=7)
    ax.set_yticks(ticks); ax.set_yticklabels(tick_lbls, fontsize=7)
    ax.set_xlabel("bin"); ax.set_ylabel("bin")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

fig54b.suptitle(
    "Task 54 — Bin-bin covariance: training vs val vs conditional vs unconditional\n"
    "HEADLINE: conditional covariance should match val better than unconditional",
    fontsize=9
)
plt.tight_layout()
plt.show()

# ── Figure C: per-bin KS tests ────────────────────────────────────────────
ks_cond  = [stats.ks_2samp(val_residuals_phys[:,k],
                             samples_cond[:,k]).statistic
            for k in range(15)]
ks_unc   = [stats.ks_2samp(val_residuals_phys[:,k],
                             samples_uncond[:,k]).statistic
            for k in range(15)]

fig54c, ax54c = plt.subplots(figsize=(12, 4))
w_ks = BIN_WIDTH * 0.35
ax54c.bar(BIN_CENTERS - w_ks/2, ks_cond, width=w_ks,
          color="tab:green", alpha=0.75, label="Conditional vs val")
ax54c.bar(BIN_CENTERS + w_ks/2, ks_unc,  width=w_ks,
          color="tab:red",   alpha=0.75, label="Unconditional vs val")
ax54c.axhline(0, color="black", linewidth=0.7)
ax54c.set_xlabel("|latitude| (°)")
ax54c.set_ylabel("KS statistic (lower = more similar to val)")
ax54c.set_title("Per-bin KS distance from val held-out distribution\n"
                "Lower green = conditional model matches val better in that bin")
ax54c.legend(fontsize=9)
ax54c.set_xlim(0, 45)
plt.tight_layout()
plt.show()

# ── Summary table ─────────────────────────────────────────────────────────
def frob(a, b):
    return float(np.linalg.norm(a - b, "fro"))

comparison_df = pd.DataFrame([
    {
        "row"                      : "val (held-out)",
        "mean_MSE_vs_val"          : 0.0,
        "std_MSE_vs_val"           : 0.0,
        "cov_Frobenius_vs_val"     : 0.0,
        "mean_KS_vs_val"           : 0.0,
    },
    {
        "row"                      : "conditional samples",
        "mean_MSE_vs_val"          : float(np.mean((m_cond - m_val)**2)),
        "std_MSE_vs_val"           : float(np.mean((s_cond - s_val)**2)),
        "cov_Frobenius_vs_val"     : frob(c_cond, c_val),
        "mean_KS_vs_val"           : float(np.mean(ks_cond)),
    },
    {
        "row"                      : "unconditional samples",
        "mean_MSE_vs_val"          : float(np.mean((m_unc  - m_val)**2)),
        "std_MSE_vs_val"           : float(np.mean((s_unc  - s_val)**2)),
        "cov_Frobenius_vs_val"     : frob(c_unc,  c_val),
        "mean_KS_vs_val"           : float(np.mean(ks_unc)),
    },
])

print("\n" + "="*75)
print("  Task 54 Summary Table")
print("="*75)
print(comparison_df.to_string(index=False))
print("="*75)

# ── Verdict ────────────────────────────────────────────────────────────────
cond_mse  = float(np.mean((m_cond - m_val)**2))
unc_mse   = float(np.mean((m_unc  - m_val)**2))
cond_frob = frob(c_cond, c_val)
unc_frob  = frob(c_unc,  c_val)
cond_ks   = float(np.mean(ks_cond))
unc_ks    = float(np.mean(ks_unc))

print("\n── Verdict ──────────────────────────────────────────────────────────")
wins = sum([cond_mse  < unc_mse,
            cond_frob < unc_frob,
            cond_ks   < unc_ks])

if wins == 3:
    verdict = ("CONDITIONING HELPS on all three metrics.\n"
               "  Conditional model tracks the val distribution better than "
               "unconditional\n  on mean, covariance, and per-bin KS distance.")
elif wins == 2:
    verdict = ("CONDITIONING HELPS on 2/3 metrics — modest but real improvement.\n"
               "  Check the covariance heatmaps to see where the gain comes from.")
elif wins == 1:
    verdict = ("MIXED — conditioning helps on only 1/3 metrics.\n"
               "  Either the dataset is too small for conditioning to matter,\n"
               "  or the model needs more training / capacity.")
else:
    verdict = ("CONDITIONING NOT HELPING — unconditional is as good or better.\n"
               "  Re-run Task 49 cond-sensitivity check; check that cond is\n"
               "  not being dropped somewhere in the forward pass.")

print(f"  {verdict}")
print(f"\n  Cond  mean_MSE : {cond_mse:.6f}  {'<' if cond_mse < unc_mse else '>='} "
      f"unc {unc_mse:.6f}")
print(f"  Cond  cov_Frob : {cond_frob:.4f}   {'<' if cond_frob < unc_frob else '>='} "
      f"unc {unc_frob:.4f}")
print(f"  Cond  mean_KS  : {cond_ks:.4f}    {'<' if cond_ks < unc_ks else '>='} "
      f"unc {unc_ks:.4f}")

print(f"\n✓ Tasks 53 and 54 complete")

---
## Task 54 — Conditional distributional verification

Headline of the evaluation notebook. For each validation window, sample
`N_PER_WIN` conditional residuals at that window's conditioning, then
aggregate over all validation windows to form a "conditional sample
distribution" comparable to the actual held-out validation residual
distribution. The unconditional Week 08 model is the reference baseline:
sample `N_TOTAL_UNCOND ≈ N_PER_WIN × N_VAL_WINDOWS` unconditional
residuals (with no targeting at all) and treat that as the "what if we
ignored conditioning" comparator.

The diagnostics from Week 08's Task 43 — bin-wise mean, bin-wise std,
bin-bin covariance heatmap — applied three ways: training residuals,
conditional samples (per-window targeting), unconditional samples
(random). The expected reading:

- *If conditioning is working*, the conditional samples' bin-wise std
  should be **smaller** than the unconditional samples' (because each
  window's conditional distribution is narrower than the marginal), and
  the conditional samples' aggregate covariance should match the
  validation residuals' aggregate covariance *better* than the
  unconditional samples' does. The improvement may be small at this
  dataset scale — the NLL notebook will quantify it.
- *If conditioning is not working*, the conditional and unconditional
  comparisons will look nearly identical. That's the same failure-mode
  signature the cond-sensitivity check at the top of this notebook
  is designed to catch *before* you spend time here.
- *If conditioning is working but the trained model is overconfident*,
  the conditional std will be visibly *narrower* than the validation
  residuals' actual per-window variability — the model produces
  residuals that look like noise-free predictions when in fact every
  real window has a genuine spread of residuals around the conditional
  mean. This is real and worth flagging if you see it.


In [ ]:
# Task 54: Distributional comparison / verification
# Depends on:
#   lightning_cond, sample_conditional,
#   windows_df, BIN_CENTERS, BIN_WIDTH,
#   DEVICE, DiffusionLightning, DiffusionMLP,
#   sample, alpha_np, sigma_np, T

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy import stats
from einops import repeat

HIST_COLS = [f"hist_emp_{j:02d}" for j in range(15)]
PAR_COLS  = [f"hist_par_{j:02d}" for j in range(15)]

# Ensure model is ready
lightning_cond.eval()
lightning_cond.to(DEVICE)

val_df   = windows_df[windows_df["split"] == "val"].reset_index(drop=True)
train_df = windows_df[windows_df["split"] == "train"].reset_index(drop=True)

print(f"Val windows  : {len(val_df)}")
print(f"Train windows: {len(train_df)}")

# ══════════════════════════════════════════════════════════════════════════
# TASK 54 — Distributional comparison
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "═"*60)
print("TASK 54 — Distributional comparison")
print("═"*60)

N_PER_WIN = 20
N_TOTAL = len(val_df) * N_PER_WIN

# ── Training residuals ────────────────────────────────────────────────────
emp_tr = train_df[HIST_COLS].to_numpy(np.float32)
par_tr = train_df[PAR_COLS].to_numpy(np.float32)

train_residuals_phys = emp_tr - par_tr

rng_54 = np.random.default_rng(123)

train_arr = train_residuals_phys[
    rng_54.integers(
        0,
        len(train_residuals_phys),
        size=N_TOTAL
    )
]

# ── Validation residuals ──────────────────────────────────────────────────
emp_v = val_df[HIST_COLS].to_numpy(np.float32)
par_v = val_df[PAR_COLS].to_numpy(np.float32)

val_residuals_phys = emp_v - par_v

# ── Conditional samples ───────────────────────────────────────────────────
cond_raw_54 = torch.tensor(
    val_df[["area_smoothed", "mu_universal"]].to_numpy(np.float32),
    device=DEVICE
)

cond_norm_54 = (
    (cond_raw_54 - lightning_cond.cond_means)
    / lightning_cond.cond_stds
)

cond_batched_54 = repeat(
    cond_norm_54,
    "n d -> (n k) d",
    k=N_PER_WIN
)

torch.manual_seed(0)

samples_cond = sample_conditional(
    lightning_cond,
    cond_batched_54,
    device=DEVICE
)

print(f"\nConditional samples generated: {samples_cond.shape}")

# ── Unconditional baseline ────────────────────────────────────────────────
CKPT_UNCOND = "./ckpt_full.ckpt"

try:
    lightning_uncond = DiffusionLightning.load_from_checkpoint(
        CKPT_UNCOND,
        model=DiffusionMLP(use_timestep_embedding=True),
        alpha=alpha_np,
        sigma=sigma_np,
        T=T,
    )

    lightning_uncond.eval()
    lightning_uncond.to(DEVICE)

    torch.manual_seed(0)

    samples_uncond = sample(
        lightning_uncond,
        batch_size=N_TOTAL,
        data_dim=15,
        device=DEVICE
    ).cpu().numpy()

    print(
        f"Unconditional samples from ckpt_full.ckpt: "
        f"{samples_uncond.shape}"
    )

except Exception as e:

    print(f"Could not load unconditional model ({e}).")
    print("Using training residuals resampled as unconditional baseline.")

    samples_uncond = train_residuals_phys[
        rng_54.integers(
            0,
            len(train_residuals_phys),
            size=N_TOTAL
        )
    ]

# ── Statistics helper ─────────────────────────────────────────────────────
def binwise_stats(arr):
    return (
        arr.mean(axis=0),
        arr.std(axis=0),
        np.cov(arr.T)
    )

m_train, s_train, c_train = binwise_stats(train_arr)
m_val,   s_val,   c_val   = binwise_stats(val_residuals_phys)
m_cond,  s_cond,  c_cond  = binwise_stats(samples_cond)
m_unc,   s_unc,   c_unc   = binwise_stats(samples_uncond)

# ── Figure A: mean + std comparison ──────────────────────────────────────
fig54a, axes54a = plt.subplots(1, 2, figsize=(15, 5))

w = BIN_WIDTH * 0.20

bar_configs = [
    (m_train, s_train, "Training",       "tab:blue"),
    (m_val,   s_val,   "Val (held-out)", "tab:purple"),
    (m_cond,  s_cond,  "Conditional",    "tab:green"),
    (m_unc,   s_unc,   "Unconditional",  "tab:red"),
]

offsets = [-1.5*w, -0.5*w, 0.5*w, 1.5*w]

for ax, stat_idx, title, ylabel in [
    (axes54a[0], 0, "Bin-wise mean", "Mean residual density"),
    (axes54a[1], 1, "Bin-wise std",  "Std residual density")
]:

    for (m, s, label, color), offset in zip(bar_configs, offsets):

        vals = m if stat_idx == 0 else s

        ax.bar(
            BIN_CENTERS + offset,
            vals,
            width=w,
            label=label,
            color=color,
            alpha=0.75,
            edgecolor="white",
            linewidth=0.5
        )

    ax.axhline(0, color="black", linewidth=0.7)

    ax.set_title(title, fontsize=10)
    ax.set_xlabel("|latitude| (°)")
    ax.set_ylabel(ylabel)

    ax.legend(fontsize=7.5)
    ax.set_xlim(0, 45)

fig54a.suptitle(
    "Task 54 — Bin-wise mean and std comparison",
    fontsize=9
)

plt.tight_layout()
plt.show()

# ── Figure B: covariance heatmaps ─────────────────────────────────────────
vmax = max(
    np.abs(c_train).max(),
    np.abs(c_val).max(),
    np.abs(c_cond).max(),
    np.abs(c_unc).max()
)

norm_cov = mcolors.TwoSlopeNorm(
    vmin=-vmax,
    vcenter=0,
    vmax=vmax
)

ticks = np.arange(0, 15, 3)

tick_lbls = [
    f"{BIN_CENTERS[i]:.0f}°"
    for i in ticks
]

fig54b, axes54b = plt.subplots(1, 4, figsize=(20, 5))

for ax, mat, title in [
    (axes54b[0], c_train, "Training"),
    (axes54b[1], c_val,   "Val (held-out)"),
    (axes54b[2], c_cond,  "Conditional samples"),
    (axes54b[3], c_unc,   "Unconditional samples"),
]:

    im = ax.imshow(
        mat,
        cmap="RdBu_r",
        norm=norm_cov,
        aspect="auto"
    )

    ax.set_title(title, fontsize=9)

    ax.set_xticks(ticks)
    ax.set_xticklabels(tick_lbls, fontsize=7)

    ax.set_yticks(ticks)
    ax.set_yticklabels(tick_lbls, fontsize=7)

    ax.set_xlabel("bin")
    ax.set_ylabel("bin")

    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

fig54b.suptitle(
    "Task 54 — Bin-bin covariance comparison",
    fontsize=9
)

plt.tight_layout()
plt.show()

# ── Figure C: KS tests ────────────────────────────────────────────────────
ks_cond = [
    stats.ks_2samp(
        val_residuals_phys[:, k],
        samples_cond[:, k]
    ).statistic
    for k in range(15)
]

ks_unc = [
    stats.ks_2samp(
        val_residuals_phys[:, k],
        samples_uncond[:, k]
    ).statistic
    for k in range(15)
]

fig54c, ax54c = plt.subplots(figsize=(12, 4))

w_ks = BIN_WIDTH * 0.35

ax54c.bar(
    BIN_CENTERS - w_ks/2,
    ks_cond,
    width=w_ks,
    color="tab:green",
    alpha=0.75,
    label="Conditional vs val"
)

ax54c.bar(
    BIN_CENTERS + w_ks/2,
    ks_unc,
    width=w_ks,
    color="tab:red",
    alpha=0.75,
    label="Unconditional vs val"
)

ax54c.axhline(0, color="black", linewidth=0.7)

ax54c.set_xlabel("|latitude| (°)")
ax54c.set_ylabel("KS statistic")

ax54c.set_title(
    "Per-bin KS distance from validation distribution"
)

ax54c.legend(fontsize=9)
ax54c.set_xlim(0, 45)

plt.tight_layout()
plt.show()

# ── Summary metrics ───────────────────────────────────────────────────────
def frob(a, b):
    return float(np.linalg.norm(a - b, "fro"))

comparison_df = pd.DataFrame([
    {
        "row": "val (held-out)",
        "mean_MSE_vs_val": 0.0,
        "std_MSE_vs_val": 0.0,
        "cov_Frobenius_vs_val": 0.0,
        "mean_KS_vs_val": 0.0,
    },
    {
        "row": "conditional samples",
        "mean_MSE_vs_val":
            float(np.mean((m_cond - m_val) ** 2)),
        "std_MSE_vs_val":
            float(np.mean((s_cond - s_val) ** 2)),
        "cov_Frobenius_vs_val":
            frob(c_cond, c_val),
        "mean_KS_vs_val":
            float(np.mean(ks_cond)),
    },
    {
        "row": "unconditional samples",
        "mean_MSE_vs_val":
            float(np.mean((m_unc - m_val) ** 2)),
        "std_MSE_vs_val":
            float(np.mean((s_unc - s_val) ** 2)),
        "cov_Frobenius_vs_val":
            frob(c_unc, c_val),
        "mean_KS_vs_val":
            float(np.mean(ks_unc)),
    },
])

print("\n" + "=" * 75)
print("Task 54 Summary Table")
print("=" * 75)

print(comparison_df.to_string(index=False))

print("=" * 75)

# ── Verdict ───────────────────────────────────────────────────────────────
cond_mse = float(np.mean((m_cond - m_val) ** 2))
unc_mse  = float(np.mean((m_unc  - m_val) ** 2))

cond_frob = frob(c_cond, c_val)
unc_frob  = frob(c_unc, c_val)

cond_ks = float(np.mean(ks_cond))
unc_ks  = float(np.mean(ks_unc))

wins = sum([
    cond_mse < unc_mse,
    cond_frob < unc_frob,
    cond_ks < unc_ks
])

print("\n── Verdict ──────────────────────────────────────────")

if wins == 3:

    verdict = (
        "CONDITIONING HELPS on all three metrics.\n"
        "Conditional model tracks validation distribution "
        "better than unconditional."
    )

elif wins == 2:

    verdict = (
        "CONDITIONING HELPS on 2/3 metrics.\n"
        "Improvement is modest but measurable."
    )

elif wins == 1:

    verdict = (
        "MIXED RESULT — conditioning helps on only 1/3 metrics."
    )

else:

    verdict = (
        "CONDITIONING NOT HELPING — unconditional performs similarly "
        "or better."
    )

print(verdict)

print(
    f"\nCond mean_MSE : {cond_mse:.6f} "
    f"{'<' if cond_mse < unc_mse else '>='} "
    f"unc {unc_mse:.6f}"
)

print(
    f"Cond cov_Frob : {cond_frob:.4f} "
    f"{'<' if cond_frob < unc_frob else '>='} "
    f"unc {unc_frob:.4f}"
)

print(
    f"Cond mean_KS  : {cond_ks:.4f} "
    f"{'<' if cond_ks < unc_ks else '>='} "
    f"unc {unc_ks:.4f}"
)

print("\n✓ Task 54 complete")

---
## Where Week 09 leaves us, and what the NLL notebook will measure

By the end of this notebook you have:

- A trained **conditional** diffusion model whose samples are visibly
  sensitive to the conditioning (Task 53) and that aggregates, at the
  distributional level, to something closer to held-out validation
  residuals than the unconditional baseline does (Task 54). The
  improvement may be small at this scale; the qualitative direction is
  what matters before quantification.
- A loaded, sanity-checked conditional checkpoint that the NLL notebook
  will use to compute the headline value-add metric.

What you have **not** done is the headline measurement: does combining
the classical ButterflAI density with conditional diffusion residuals
actually outperform the classical density alone, on `compute_global_nll`
applied to the held-out validation split? That comparison is its own
clean notebook, picking up where this one ends.

**What the NLL notebook will do.** For each validation window: read its
`(area, mu)`, build the classical density on `BIN_CENTERS`, draw N
conditional residual samples at that conditioning, add the samples to
the classical density (with appropriate non-negativity / normalization
handling), compute the per-window NLL of the actual emp histogram under
that mixture. Aggregate via `compute_global_nll` across all validation
windows. Compare against (a) classical alone and (b) classical + Week 08
unconditional samples to isolate how much the conditioning is worth.

That's the end-of-program payoff measurement. Everything in Week 09 was
infrastructure pointed at making that measurement possible.
